# 01 · Preprocessing

**Goal of the project:** when a satellite first sees a new California fire, predict whether it will grow past **1,000 acres**.

**Goal of this notebook:** turn `training_table.csv` into clean model inputs (`X`) and answers (`y`) for train, validation and test, without leaking information from the future or from the test years.

Plan:
1. Load the table and look at it
2. Separate inputs, label and reference columns
3. Split by year
4. Fill the few missing values
5. Encode the text and circular columns
6. Handle skewed numbers
7. Save the result

## Step 1 · Load the table and look at it

Before changing anything, we check what we actually have: how many rows and columns, what types they are, how balanced the label is, and where values are missing.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

df = pd.read_csv("../data/processed/training_table.csv", parse_dates=["first_detect_local"])
print(df.shape)
df.head()

(5565, 55)


,event_id,fod_id,fire_name,complex_name,first_detect_local,year,split,hard_case,detect_lag_days,match_km,fire_size,fire_size_class,nwcg_general_cause,post_n_detections,fp_n_pixels,fp_frp_sum,fp_frp_max,fp_frp_mean,fp_bright_ti4_max,fp_bright_ti4_mean,fp_bright_ti5_mean,fp_scan_mean,fp_track_mean,fp_share_high_conf,is_night,is_noaa20,tmax_c_prev,rhmin_prev,wind_ms_prev,wind_dir_prev,vpd_prev,fm100_prev,fm1000_prev,erc_prev,bi_prev,erc_7d_mean,vpd_7d_mean,tmax_c_7d_mean,wind_ms_7d_max,precip_mm_30d,days_since_rain,ndvi,fbfm40,fuel_family,elevation_m,slope_deg,aspect_northness,aspect_eastness,lat,lon,month,doy_sin,doy_cos,local_hour,label_big_fire
0,4,201644970,24,NaN,2012-01-21 12:44:00,2012,train,1,2,2.768936,0.35,B,Debris and open burning,2,2,9.99,5.10,4.995,329.63,328.925,284.490,0.44,0.39,0.0,0,0,10.35,38.2,2.8,48.0,0.49,12.3,16.0,27.0,0.0,36.428571,0.545714,11.764286,3.7,3.3,1,0.3480,102.0,grass,89.0,0.0,0.000000,0.000000,37.075385,-120.091235,1,0.353445,0.935455,12.733333,0
1,101,201642448,MORELLI LN OCCIDENTAL,NaN,2012-02-04 13:22:00,2012,train,1,0,2.424449,1.00,B,Debris and open burning,1,1,16.17,16.17,16.170,332.01,332.010,293.590,0.45,0.39,0.0,0,0,16.15,44.2,1.8,143.0,0.51,18.9,23.3,16.0,16.0,12.857143,0.388571,15.092857,4.1,238.0,12,0.5963,102.0,grass,67.0,8.0,-0.121869,0.992546,38.445060,-122.913770,2,0.566362,0.824157,13.366667,0
2,163,201644983,HIDDEN,NaN,2012-02-09 13:28:00,2012,train,1,1,4.860152,2.00,B,Debris and open burning,2,2,8.44,4.56,4.220,339.41,339.025,296.495,0.48,0.48,0.0,0,0,16.75,39.7,1.7,358.0,0.53,13.3,15.7,37.0,25.0,35.714286,0.685714,16.550000,5.5,86.3,17,0.3957,102.0,grass,618.0,10.0,-0.559193,-0.829038,37.316210,-119.797980,2,0.635068,0.772456,13.466667,0
3,291,201655412,JACK SLOUGH RD / WOODRUFF,NaN,2012-02-22 12:45:00,2012,train,1,0,4.249543,0.01,A,Missing data/not specified/undetermined,1,1,15.86,15.86,15.860,331.77,331.770,295.950,0.46,0.39,0.0,0,0,21.85,32.9,2.3,328.0,0.85,13.7,16.7,36.0,30.0,33.714286,0.728571,17.978571,4.4,45.3,9,0.2951,102.0,grass,24.0,0.0,0.000000,0.000000,39.248810,-121.579360,2,0.790563,0.612380,12.750000,0
4,308,201642453,SODA,NaN,2012-02-23 12:26:00,2012,train,1,0,1.701480,200.00,D,Debris and open burning,4,2,42.48,21.24,21.240,338.27,334.570,295.235,0.58,0.52,0.0,0,0,22.75,29.4,4.1,341.0,1.09,13.2,17.2,35.0,33.0,31.428571,0.697143,16.792857,5.0,51.0,10,0.6351,142.0,shrub,287.0,28.0,0.615661,-0.788011,38.436845,-122.319925,2,0.800980,0.598691,12.433333,0


**What to look for:** one row per fire. You should see 5,565 rows and 55 columns. Scroll right in the preview: satellite columns start with `fp_`, then weather, fuel, terrain, and the label at the far right.

In [2]:
# How many fires became big?
counts = df["label_big_fire"].value_counts()
counts

label_big_fire
0    5078
1     487
Name: count, dtype: int64

In [3]:
counts = df["label_big_fire"].value_counts(normalize=True)
counts

label_big_fire
0    0.912489
1    0.087511
Name: proportion, dtype: float64

**Why this matters:** only about 1 in 11 fires becomes big. A model that always says "small" would be 91% accurate and completely useless. So **accuracy is the wrong score** here; later we'll use PR-AUC and F1, which focus on the big fires.

In [4]:
# Column types: numbers vs text vs dates
df.dtypes.value_counts()

float64           36
int64             12
str                6
datetime64[us]     1
Name: count, dtype: int64

In [5]:
# Which columns are text? These need special handling before a model can use them.
df.select_dtypes(include="str").columns

Index(['fire_name', 'complex_name', 'split', 'fire_size_class',
       'nwcg_general_cause', 'fuel_family'],
      dtype='str')

**What to look for:** most columns are numbers (`float64`, `int64`). Text columns show up as `str`. The text columns are either reference info (fire name, cause, size class, split) or one real input: `fuel_family`. Models only understand numbers, so `fuel_family` will need encoding in Step 5.

In [6]:
# Missing values: which columns, and how many?
missing = df.isna().sum()
missing[missing>0]

fire_name         872
complex_name     5367
erc_prev            1
erc_7d_mean         1
precip_mm_30d       2
ndvi                6
dtype: int64

**What to look for:** `complex_name` is mostly empty because most fires aren't part of a complex, and `fire_name` is empty for small unnamed fires. Both are reference columns, so it doesn't matter. Among the **inputs**, only a handful of values are missing (a few weather and NDVI values). We'll fill those in Step 4.

In [7]:
# Quick summary of a few important inputs
df[["fp_n_pixels", "fp_frp_sum", "erc_prev", "rhmin_prev", "wind_ms_prev", "ndvi", "slope_deg"]].describe().round(3)

,fp_n_pixels,fp_frp_sum,erc_prev,rhmin_prev,wind_ms_prev,ndvi,slope_deg
count,5565.000,5565.000,5564.000,5565.000,5565.000,5559.000,5565.000
mean,4.364,112.441,65.314,21.475,3.350,0.432,8.959
std,23.330,925.887,18.166,11.775,1.432,0.186,10.052
min,1.000,0.100,2.000,1.000,0.600,0.020,0.000
25%,1.000,2.150,53.000,13.100,2.400,0.284,1.000
50%,1.000,6.450,67.000,19.400,3.100,0.407,5.000
75%,2.000,24.040,79.000,27.300,4.000,0.568,15.000
max,826.000,40724.460,110.000,97.400,12.500,0.925,62.000


**What to look for:** compare the **mean** and the **max**. For `fp_n_pixels` and `fp_frp_sum`, the max is huge compared with the median (the 50% row). A few giant fires stretch the scale. That's called a **skewed** distribution, and we'll deal with it in Step 6.

---
**Stop here.** Run every cell above, look at the outputs, and tell me what you notice or what's unclear. Then we'll do Step 2.

step 2

In [8]:
LABEL = "label_big_fire"

# Columns the model must NOT see:
# - info about what happened after the fire started (final size, total detections) -> that's cheating ("leakage")
# - IDs and names
META = ["event_id", "fod_id", "fire_name", "complex_name", "first_detect_local", "fire_size", "fire_size_class",
        "nwcg_general_cause", "post_n_detections", "match_km", "detect_lag_days"]

# Columns we use for splitting/evaluating, not as inputs
HELPERS = ["split", "hard_case", "year"]

# feature_cols = every column that isn't the label, META or HELPERS (list comprehension over df.columns)
# should come out to 40
feature_cols = []
for col in df.columns:
    if (col not in META and col not in HELPERS and col != LABEL):
        feature_cols.append(col)

X = df[feature_cols]
y = df[LABEL]

df.head()
# X = df with only the feature columns, y = the label column


,event_id,fod_id,fire_name,complex_name,first_detect_local,year,split,hard_case,detect_lag_days,match_km,fire_size,fire_size_class,nwcg_general_cause,post_n_detections,fp_n_pixels,fp_frp_sum,fp_frp_max,fp_frp_mean,fp_bright_ti4_max,fp_bright_ti4_mean,fp_bright_ti5_mean,fp_scan_mean,fp_track_mean,fp_share_high_conf,is_night,is_noaa20,tmax_c_prev,rhmin_prev,wind_ms_prev,wind_dir_prev,vpd_prev,fm100_prev,fm1000_prev,erc_prev,bi_prev,erc_7d_mean,vpd_7d_mean,tmax_c_7d_mean,wind_ms_7d_max,precip_mm_30d,days_since_rain,ndvi,fbfm40,fuel_family,elevation_m,slope_deg,aspect_northness,aspect_eastness,lat,lon,month,doy_sin,doy_cos,local_hour,label_big_fire
0,4,201644970,24,NaN,2012-01-21 12:44:00,2012,train,1,2,2.768936,0.35,B,Debris and open burning,2,2,9.99,5.10,4.995,329.63,328.925,284.490,0.44,0.39,0.0,0,0,10.35,38.2,2.8,48.0,0.49,12.3,16.0,27.0,0.0,36.428571,0.545714,11.764286,3.7,3.3,1,0.3480,102.0,grass,89.0,0.0,0.000000,0.000000,37.075385,-120.091235,1,0.353445,0.935455,12.733333,0
1,101,201642448,MORELLI LN OCCIDENTAL,NaN,2012-02-04 13:22:00,2012,train,1,0,2.424449,1.00,B,Debris and open burning,1,1,16.17,16.17,16.170,332.01,332.010,293.590,0.45,0.39,0.0,0,0,16.15,44.2,1.8,143.0,0.51,18.9,23.3,16.0,16.0,12.857143,0.388571,15.092857,4.1,238.0,12,0.5963,102.0,grass,67.0,8.0,-0.121869,0.992546,38.445060,-122.913770,2,0.566362,0.824157,13.366667,0
2,163,201644983,HIDDEN,NaN,2012-02-09 13:28:00,2012,train,1,1,4.860152,2.00,B,Debris and open burning,2,2,8.44,4.56,4.220,339.41,339.025,296.495,0.48,0.48,0.0,0,0,16.75,39.7,1.7,358.0,0.53,13.3,15.7,37.0,25.0,35.714286,0.685714,16.550000,5.5,86.3,17,0.3957,102.0,grass,618.0,10.0,-0.559193,-0.829038,37.316210,-119.797980,2,0.635068,0.772456,13.466667,0
3,291,201655412,JACK SLOUGH RD / WOODRUFF,NaN,2012-02-22 12:45:00,2012,train,1,0,4.249543,0.01,A,Missing data/not specified/undetermined,1,1,15.86,15.86,15.860,331.77,331.770,295.950,0.46,0.39,0.0,0,0,21.85,32.9,2.3,328.0,0.85,13.7,16.7,36.0,30.0,33.714286,0.728571,17.978571,4.4,45.3,9,0.2951,102.0,grass,24.0,0.0,0.000000,0.000000,39.248810,-121.579360,2,0.790563,0.612380,12.750000,0
4,308,201642453,SODA,NaN,2012-02-23 12:26:00,2012,train,1,0,1.701480,200.00,D,Debris and open burning,4,2,42.48,21.24,21.240,338.27,334.570,295.235,0.58,0.52,0.0,0,0,22.75,29.4,4.1,341.0,1.09,13.2,17.2,35.0,33.0,31.428571,0.697143,16.792857,5.0,51.0,10,0.6351,142.0,shrub,287.0,28.0,0.615661,-0.788011,38.436845,-122.319925,2,0.800980,0.598691,12.433333,0


step 3

In [9]:
## Step 3 · Split by year

# The "split" column already says which years go where:
# train = 2012-2020, val = 2021-2022, test = 2023-2024
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

# Inputs (X) and answers (y) for each split, using the columns from Step 2
X_train, y_train = train_df[feature_cols], train_df[LABEL]
X_val, y_val = val_df[feature_cols], val_df[LABEL]
X_test, y_test = test_df[feature_cols], test_df[LABEL]

# Size of each split and how many of its fires became big
for name, X_part, y_part in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5}  {X_part.shape}  big fires: {y_part.sum()} ({y_part.mean():.1%})")

train  (3603, 40)  big fires: 368 (10.2%)
val    (938, 40)  big fires: 53 (5.7%)
test   (1024, 40)  big fires: 66 (6.4%)


In [10]:
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_val = X_val.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))


In [11]:
X_train.isna().sum().sum()

np.int64(0)

In [12]:
from sklearn.preprocessing import OneHotEncoder
fuel_encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False)
fuel_encoder.fit(X_train[["fuel_family"]])
fuel_encoder.get_feature_names_out()

array(['fuel_family_grass', 'fuel_family_grass_shrub',
       'fuel_family_nonburnable', 'fuel_family_shrub',
       'fuel_family_timber_litter', 'fuel_family_timber_understory',
       'fuel_family_infrequent_sklearn'], dtype=object)

In [13]:
def encode (X2):
    X2 = X2.copy()
    X2["wind_dir_sin"] = np.sin(np.radians(X2["wind_dir_prev"]))
    X2["wind_dir_cos"] = np.cos(np.radians(X2["wind_dir_prev"]))
    fuel = pd.DataFrame(fuel_encoder.transform(X2[["fuel_family"]]),
                        columns=fuel_encoder.get_feature_names_out(), index=X2.index)
    X2 = X2.drop(columns=["wind_dir_prev", "fbfm40", "fuel_family"])
    return pd.concat([X2, fuel], axis=1)

X_train = encode(X_train)
X_val = encode(X_val)
X_test = encode(X_test)

print (X_train.shape, X_val.shape, X_test.shape)


(3603, 46) (938, 46) (1024, 46)


In [14]:
## Step 6 · Skewed numbers and scaling

from sklearn.preprocessing import StandardScaler

# 1) Log transform: compresses the few huge values (log1p because rain can be 0)
SKEWED = ["fp_frp_sum", "fp_n_pixels", "fp_frp_max", "precip_mm_30d", "fp_frp_mean"]

X_train[SKEWED] = np.log1p(X_train[SKEWED])
X_val[SKEWED] = np.log1p(X_val[SKEWED])
X_test[SKEWED] = np.log1p(X_test[SKEWED])


print(X_train[SKEWED].skew().round(1))

# 2) Scaling, a SEPARATE copy for logistic regression / TabNet (XGBoost uses the unscaled X_train)
# fit_transform on train, only transform on val/test (same rule as the encoder)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_train.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns, index=X_test.index)

print(X_train_scaled.mean().abs().max().round(3), X_train_scaled.std().mean().round(2))

fp_frp_sum       1.2
fp_n_pixels      2.8
fp_frp_max       1.0
precip_mm_30d    0.8
fp_frp_mean      0.8
dtype: float64
0.0 1.0


In [16]:
## Step 7 · Save the model-ready data

import joblib
from pathlib import Path

OUT = Path("../data/processed/model_ready")
OUT.mkdir(exist_ok=True)

# Info we need later for evaluation but NOT as model inputs:
# hard_case (to score the hard fires separately), plus names/sizes to look at mistakes
INFO = ["event_id", "year", "hard_case", "fire_name", "fire_size"]

for name, X_part, X_part_scaled, y_part, part_df in [
    ("train", X_train, X_train_scaled, y_train, train_df),
    ("val", X_val, X_val_scaled, y_val, val_df),
    ("test", X_test, X_test_scaled, y_test, test_df),
]:
    X_part.to_csv(OUT / f"X_{name}.csv", index=False)                # for XGBoost, CatBoost, EBM
    X_part_scaled.to_csv(OUT / f"X_{name}_scaled.csv", index=False)  # for logistic regression
    y_part.to_csv(OUT / f"y_{name}.csv", index=False)
    part_df[INFO].to_csv(OUT / f"info_{name}.csv", index=False)

# Save the fitted encoder and scaler, so new fires can be processed exactly the same way later
joblib.dump({"fuel_encoder": fuel_encoder, "scaler": scaler, "train_medians": X_train.median(numeric_only=True),
             "skewed": SKEWED, "feature_cols": list(X_train.columns)}, OUT / "preprocessing.joblib")

print(sorted(p.name for p in OUT.iterdir()))

['X_test.csv', 'X_test_scaled.csv', 'X_train.csv', 'X_train_scaled.csv', 'X_val.csv', 'X_val_scaled.csv', 'info_test.csv', 'info_train.csv', 'info_val.csv', 'preprocessing.joblib', 'y_test.csv', 'y_train.csv', 'y_val.csv']
